# 04 Lag Analysis

Find the strongest statistically significant search-to-sales lag, build the headline heatmap, and store results in DuckDB.

In [ ]:
from pathlib import Path
import pickle

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns


def find_project_root() -> Path:
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DASHBOARD_DIR = PROJECT_ROOT / "dashboard"
DATA_DIR = PROJECT_ROOT / "data"

DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

cleaned_pairs_path = PROCESSED_DIR / "cleaned_pairs.pkl"
census_clean_path = PROCESSED_DIR / "census_clean.csv"

if not cleaned_pairs_path.exists():
    raise FileNotFoundError(f"Missing {cleaned_pairs_path}. Run 03_seasonality.ipynb first.")
if not census_clean_path.exists():
    raise FileNotFoundError(f"Missing {census_clean_path}. Run 01_data_collection.ipynb first.")

with open(cleaned_pairs_path, "rb") as f:
    cleaned_pairs = pickle.load(f)

census_df = pd.read_csv(census_clean_path, parse_dates=["date"])

print("Cleaned categories:", sorted(cleaned_pairs))
print("Census rows:", len(census_df))

In [ ]:
def lag_with_significance(df, max_lag=12, min_obs=8):
    results = []

    for lag in range(0, max_lag + 1):
        if lag == 0:
            search = df["search_clean"].values
            sales = df["sales_clean"].values
        else:
            # Search at time t vs sales at time t + lag.
            search = df["search_clean"].values[:-lag]
            sales = df["sales_clean"].values[lag:]

        min_len = min(len(search), len(sales))
        search = search[-min_len:]
        sales = sales[-min_len:]

        mask = ~(np.isnan(search) | np.isnan(sales))
        search = search[mask]
        sales = sales[mask]
        n = len(search)

        if n < min_obs or n <= 3:
            continue

        corr, pvalue = stats.pearsonr(search, sales)

        # Fisher z confidence interval for Pearson correlation.
        corr_for_ci = np.clip(corr, -0.999999, 0.999999)
        z_corr = np.arctanh(corr_for_ci)
        stderr = 1.0 / np.sqrt(n - 3)
        delta = 1.96 * stderr
        ci_lower = np.tanh(z_corr - delta)
        ci_upper = np.tanh(z_corr + delta)

        results.append({
            "lag_months": lag,
            "n": n,
            "correlation": corr,
            "pvalue": pvalue,
            "significant": pvalue < 0.05,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "ci_excludes_zero": (ci_lower > 0) or (ci_upper < 0),
        })

    return pd.DataFrame(results)


def find_optimal_lag(df, max_lag=12):
    return lag_with_significance(df, max_lag=max_lag)

In [ ]:
all_results = {}
summary_rows = []
detailed_rows = []

for category, df in cleaned_pairs.items():
    print(f"\n{'=' * 40}")
    print(f"Category: {category}")
    print(f"{'=' * 40}")

    lag_df = lag_with_significance(df)
    lag_df.insert(0, "category", category)
    all_results[category] = lag_df
    detailed_rows.append(lag_df)

    display_cols = [
        "lag_months",
        "n",
        "correlation",
        "pvalue",
        "significant",
        "ci_lower",
        "ci_upper",
        "ci_excludes_zero",
    ]
    print(lag_df[display_cols].round(4).to_string(index=False))

    defensible = lag_df[lag_df["significant"] & lag_df["ci_excludes_zero"]]

    if defensible.empty:
        print("  No statistically defensible lag found")
        optimal_lag = np.nan
        peak_corr = np.nan
        pvalue = np.nan
        ci_lower = np.nan
        ci_upper = np.nan
        n = np.nan
    else:
        best_row = defensible.loc[defensible["correlation"].abs().idxmax()]
        optimal_lag = int(best_row["lag_months"])
        peak_corr = best_row["correlation"]
        pvalue = best_row["pvalue"]
        ci_lower = best_row["ci_lower"]
        ci_upper = best_row["ci_upper"]
        n = int(best_row["n"])
        print(
            f"\n  Optimal lag: {optimal_lag} months | "
            f"Correlation: {peak_corr:.4f} | p-value: {pvalue:.4g} | "
            f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]"
        )

    summary_rows.append({
        "category": category,
        "optimal_lag_months": optimal_lag,
        "peak_corr": peak_corr,
        "pvalue": pvalue,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "n": n,
        "statistically_defensible": not defensible.empty,
    })

summary_df = pd.DataFrame(summary_rows)
detailed_lag_df = pd.concat(detailed_rows, ignore_index=True)

print("\n\n=== SUMMARY ===")
print(summary_df.to_string(index=False))

summary_path = PROCESSED_DIR / "lag_summary.csv"
detailed_path = PROCESSED_DIR / "lag_results_detailed.csv"
summary_df.to_csv(summary_path, index=False)
detailed_lag_df.to_csv(detailed_path, index=False)
print("Saved:", summary_path)
print("Saved:", detailed_path)

In [ ]:
heatmap_data = pd.DataFrame()
annotation_data = pd.DataFrame()

for category, lag_df in all_results.items():
    lag_indexed = lag_df.set_index("lag_months")
    heatmap_data[category] = lag_indexed["correlation"]
    annotation_data[category] = lag_indexed.apply(
        lambda row: f"{row['correlation']:.2f}*"
        if row["significant"] and row["ci_excludes_zero"]
        else f"{row['correlation']:.2f}",
        axis=1,
    )

plt.figure(figsize=(12, 6))
sns.heatmap(
    heatmap_data.T,
    annot=annotation_data.T,
    fmt="",
    cmap="RdYlGn",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "Pearson Correlation"},
)
plt.title("Search-to-Sales Lag Correlation by Category\n(Seasonality-Adjusted)", fontsize=14)
plt.xlabel("Lag (months)")
plt.ylabel("Category")
plt.figtext(0.01, 0.01, "* p < 0.05 and 95% CI excludes zero", ha="left", fontsize=9)
plt.tight_layout(rect=[0, 0.04, 1, 1])

heatmap_path = DASHBOARD_DIR / "lag_heatmap.png"
plt.savefig(heatmap_path, dpi=150)
print("Saved:", heatmap_path)
plt.show()

In [ ]:
db_path = DATA_DIR / "search_lag.duckdb"
con = duckdb.connect(str(db_path))

category_dim = pd.DataFrame([
    {"category_id": 1, "category_name": "running_shoes", "price_tier": "medium", "purchase_type": "low-consideration"},
    {"category_id": 2, "category_name": "furniture", "price_tier": "high", "purchase_type": "high-involvement"},
    {"category_id": 3, "category_name": "laptops", "price_tier": "high", "purchase_type": "high-involvement"},
    {"category_id": 4, "category_name": "apparel", "price_tier": "low", "purchase_type": "low-consideration"},
    {"category_id": 5, "category_name": "washing_machines", "price_tier": "high", "purchase_type": "high-involvement"},
])

census_to_project = {
    "Clothing and clothing accessories stores": ["running_shoes", "apparel"],
    "Furniture and home furnishings stores": ["furniture"],
    "Electronics and appliance stores": ["laptops", "washing_machines"],
}

retail_rows = []
for census_category, project_categories in census_to_project.items():
    category_sales = census_df[census_df["category"] == census_category]
    for project_category in project_categories:
        mapped = category_sales.copy()
        mapped["category"] = project_category
        retail_rows.append(mapped)

fct_retail_sales_df = pd.concat(retail_rows, ignore_index=True)
fct_retail_sales_df["sales_millions"] = pd.to_numeric(
    fct_retail_sales_df["sales_millions"],
    errors="coerce",
)
fct_retail_sales_df = fct_retail_sales_df.dropna(subset=["sales_millions"])

con.register("category_dim", category_dim)
con.register("summary_df", summary_df)
con.register("detailed_lag_df", detailed_lag_df)
con.register("fct_retail_sales_df", fct_retail_sales_df)

con.execute("DROP TABLE IF EXISTS dim_category")
con.execute("CREATE TABLE dim_category AS SELECT * FROM category_dim")

con.execute("DROP TABLE IF EXISTS lag_results")
con.execute("CREATE TABLE lag_results AS SELECT * FROM summary_df")

con.execute("DROP TABLE IF EXISTS lag_results_detailed")
con.execute("CREATE TABLE lag_results_detailed AS SELECT * FROM detailed_lag_df")

con.execute("DROP TABLE IF EXISTS fct_retail_sales")
con.execute("CREATE TABLE fct_retail_sales AS SELECT * FROM fct_retail_sales_df")

con.execute("DROP VIEW IF EXISTS vw_ad_timing_recommendations")
con.execute("""
CREATE VIEW vw_ad_timing_recommendations AS
WITH sales_peaks AS (
    SELECT
        category,
        EXTRACT(MONTH FROM date) AS peak_month,
        AVG(sales_millions) AS avg_sales
    FROM fct_retail_sales
    GROUP BY category, EXTRACT(MONTH FROM date)
),
peak_identified AS (
    SELECT
        category,
        peak_month,
        avg_sales,
        RANK() OVER (PARTITION BY category ORDER BY avg_sales DESC) AS sales_rank
    FROM sales_peaks
),
lag_joined AS (
    SELECT
        p.category,
        p.peak_month,
        p.avg_sales,
        l.optimal_lag_months,
        CASE
            WHEN p.peak_month - l.optimal_lag_months < 1
            THEN p.peak_month - l.optimal_lag_months + 12
            ELSE p.peak_month - l.optimal_lag_months
        END AS recommended_ramp_month,
        l.peak_corr,
        l.pvalue,
        l.ci_lower,
        l.ci_upper
    FROM peak_identified p
    JOIN lag_results l ON p.category = l.category
    WHERE p.sales_rank = 1
      AND l.statistically_defensible = TRUE
)
SELECT
    category,
    peak_month AS historical_sales_peak_month,
    optimal_lag_months AS search_to_purchase_lag_months,
    recommended_ramp_month,
    ROUND(avg_sales, 1) AS avg_peak_sales_millions,
    ROUND(peak_corr, 4) AS correlation_at_optimal_lag,
    pvalue,
    ROUND(ci_lower, 4) AS ci_lower,
    ROUND(ci_upper, 4) AS ci_upper
FROM lag_joined
ORDER BY optimal_lag_months DESC
""")

ad_recommendations_df = con.execute("SELECT * FROM vw_ad_timing_recommendations").df()
ad_recommendations_path = PROCESSED_DIR / "ad_timing_recommendations.csv"
ad_recommendations_df.to_csv(ad_recommendations_path, index=False)

print("lag_results")
print(con.execute("SELECT * FROM lag_results").df())
print("dim_category")
print(con.execute("SELECT * FROM dim_category").df())
print("ad timing recommendations")
print(ad_recommendations_df)

con.close()
print("Saved:", ad_recommendations_path)
print("Database ready:", db_path)

## Category-Tier Hypothesis Tests

Compare low-consideration categories against high-involvement categories using Welch's t-test and Mann-Whitney U. With only five category-level observations, treat these as directional diagnostics rather than definitive population-level inference.

In [ ]:
low_consideration = ["apparel", "running_shoes"]
high_involvement = ["furniture", "laptops", "washing_machines"]

tier_summary = summary_df.dropna(subset=["optimal_lag_months", "peak_corr"]).copy()
tier_summary["tier"] = np.where(
    tier_summary["category"].isin(low_consideration),
    "low_consideration",
    "high_involvement",
)


def compare_tiers(metric):
    low_values = tier_summary.loc[tier_summary["tier"] == "low_consideration", metric].astype(float)
    high_values = tier_summary.loc[tier_summary["tier"] == "high_involvement", metric].astype(float)

    t_stat, t_pvalue = stats.ttest_ind(
        low_values,
        high_values,
        equal_var=False,
        nan_policy="omit",
    )
    mw_stat, mw_pvalue = stats.mannwhitneyu(
        low_values,
        high_values,
        alternative="two-sided",
    )

    return {
        "metric": metric,
        "low_consideration_n": len(low_values),
        "high_involvement_n": len(high_values),
        "low_consideration_mean": low_values.mean(),
        "high_involvement_mean": high_values.mean(),
        "welch_t_statistic": t_stat,
        "welch_pvalue": t_pvalue,
        "mannwhitney_u_statistic": mw_stat,
        "mannwhitney_pvalue": mw_pvalue,
    }


tier_tests_df = pd.DataFrame([
    compare_tiers("optimal_lag_months"),
    compare_tiers("peak_corr"),
])

tier_tests_path = PROCESSED_DIR / "tier_hypothesis_tests.csv"
tier_tests_df.to_csv(tier_tests_path, index=False)

print("Tier assignments")
print(tier_summary[["category", "tier", "optimal_lag_months", "peak_corr"]].to_string(index=False))
print("\nTier hypothesis tests")
print(tier_tests_df.round(4).to_string(index=False))
print("\nInterpretation: these tests are useful for interview discussion, but n=5 means results should be framed as directional evidence.")
print("Saved:", tier_tests_path)

## Year-over-Year Lag Stability

Run lag analysis separately by year to see whether optimal lags are stable or disrupted over time. Each yearly slice has limited observations, so this is a stability diagnostic rather than a strong inference test.

In [ ]:
yearly_rows = []

for category, df in cleaned_pairs.items():
    for year in range(2018, 2024):
        df_year = df[df.index.year == year]
        if len(df_year) < 10:
            yearly_rows.append({
                "category": category,
                "year": year,
                "optimal_lag_months": np.nan,
                "correlation": np.nan,
                "pvalue": np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
                "n": len(df_year),
                "statistically_defensible": False,
                "note": "insufficient yearly observations",
            })
            continue

        yearly_lags = lag_with_significance(df_year, max_lag=6, min_obs=6)
        defensible = yearly_lags[yearly_lags["significant"] & yearly_lags["ci_excludes_zero"]]

        if defensible.empty:
            yearly_rows.append({
                "category": category,
                "year": year,
                "optimal_lag_months": np.nan,
                "correlation": np.nan,
                "pvalue": np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
                "n": len(df_year),
                "statistically_defensible": False,
                "note": "no p<0.05 lag with CI excluding zero",
            })
        else:
            best = defensible.loc[defensible["correlation"].abs().idxmax()]
            yearly_rows.append({
                "category": category,
                "year": year,
                "optimal_lag_months": int(best["lag_months"]),
                "correlation": best["correlation"],
                "pvalue": best["pvalue"],
                "ci_lower": best["ci_lower"],
                "ci_upper": best["ci_upper"],
                "n": int(best["n"]),
                "statistically_defensible": True,
                "note": "",
            })

yearly_stability_df = pd.DataFrame(yearly_rows)
yearly_stability_path = PROCESSED_DIR / "yearly_lag_stability.csv"
yearly_stability_df.to_csv(yearly_stability_path, index=False)

stability_heatmap = yearly_stability_df.pivot(
    index="category",
    columns="year",
    values="optimal_lag_months",
)

plt.figure(figsize=(10, 5))
sns.heatmap(
    stability_heatmap,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu",
    linewidths=0.5,
    cbar_kws={"label": "Optimal Lag (Months)"},
)
plt.title("Year-over-Year Search-to-Sales Lag Stability\n(Significant Yearly Lags Only)", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Category")
plt.tight_layout()

yearly_heatmap_path = DASHBOARD_DIR / "yearly_lag_stability.png"
plt.savefig(yearly_heatmap_path, dpi=150)
print("Saved:", yearly_stability_path)
print("Saved:", yearly_heatmap_path)
print(stability_heatmap)
plt.show()